# Phase 1 LoRA CPT — Qwen2.5-Coder-7B Base (Unsloth Colab)

Self-contained notebook (no external train script). **Text-only** dense Causal LM via `FastLanguageModel` so Unsloth sample packing stays ACTIVE.

| Item | Value |
| --- | --- |
| Model | `unsloth/Qwen2.5-Coder-7B` **Base** (not Instruct) |
| Precision | **BF16 LoRA** (`load_in_4bit=False`) |
| Task | Text CPT on `Aniket200325/coder-pretrain-60gb` |
| Token budget | ~**5B** full / 1M smoke |
| Hardware | Colab **A100 80GB** (primary); 40GB OK with lower batch |
| Full batch | **110** (80GB); lower on 40GB |
| Wall stop | **11.5 h** → save + stop before ~12h Colab limit |
| Packing | **Required** — abort if inactive |
| Resume | Drive + Hub `LATEST` / `checkpoint-*` |

Smoke first (`SMOKE=True`). Only open full CPT after the packing line prints **ACTIVE**. Quantize later for Jetson Orin Nano 8GB.


### Installation


In [ ]:
%%capture
import os, re

# BEFORE any Unsloth import — avoid fast CDN hang + stats probes
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.pop(k, None)
os.environ["PYTHONUNBUFFERED"] = "1"

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {
        "2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2", "2.11": "0.0.35",
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps transformers
    !pip install --no-deps --upgrade "torchao>=0.16.0"


In [ ]:
import os
import torch, transformers
print("transformers", transformers.__version__, "torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Download env OK:", {
    k: os.environ.get(k) for k in (
        "UNSLOTH_STABLE_DOWNLOADS", "UNSLOTH_DISABLE_STATISTICS",
        "HF_HUB_DISABLE_XET", "HF_HUB_ENABLE_HF_TRANSFER",
    )
})


### Config + auth

Fill `HF_TOKEN` (or Colab Secret). Set `SMOKE = True` for a short packing dry-run; `False` for a full CPT session.


In [ ]:
import os, json, time, shutil, math
from pathlib import Path
from dataclasses import dataclass, asdict

# ── fill these ──────────────────────────────────────────────────────────────
HF_TOKEN = ""  # or Colab Secrets / userdata
HUB_MODEL_ID = ""  # e.g. "YOUR_USER/coder-qwen25-coder-7b-phase1-lora" (empty = no Hub push)
USE_DRIVE = True
SMOKE = True  # True → short smoke; False → full session CPT

MODEL_NAME = "unsloth/Qwen2.5-Coder-7B"
DATASET = "Aniket200325/coder-pretrain-60gb"
MAX_SEQ_LEN = 2048
BATCH = 2 if SMOKE else 110  # A100 80GB: fill VRAM (~batch 32 ≈ 32GB peak); smoke stays small
ACCUM = 1
LORA_R = 64
LORA_ALPHA = 128
LR = 1e-4
SEED = 42

TOKEN_BUDGET = 1_000_000 if SMOKE else 5_000_000_000
MAX_STEPS = 50 if SMOKE else None  # None → derived from token budget
SAVE_STEPS = 25 if SMOKE else 250
LOGGING_STEPS = 1 if SMOKE else 5
CKPT_MINUTES = 5.0 if SMOKE else 30.0
PROJECT_AFTER_MINUTES = 2.0 if SMOKE else 10.0
REMAINING_COLAB_HOURS = 45.0
WALL_CLOCK_HOURS = 11.5  # stop before Colab ~12h kill; 0 = disable
PACKING = True
RESUME = "none" if SMOKE else "auto"

OUT_DIR = Path("/content/ckpts/qwen25-coder-7b-phase1-lora-smoke" if SMOKE else "/content/ckpts/qwen25-coder-7b-phase1-lora")
MODEL_DIR = Path("/content/models") / MODEL_NAME.replace("/", "--")
# ────────────────────────────────────────────────────────────────────────────

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")

assert HF_TOKEN, "Set HF_TOKEN (Hugging Face write/read token)"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

DRIVE_CKPT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CKPT = Path("/content/drive/MyDrive/coder-qwen25-coder-7b-phase1-lora")
    DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Throughput / Ampere knobs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("SMOKE:", SMOKE, "| OUT:", OUT_DIR, "| DRIVE:", DRIVE_CKPT)
print("batch", BATCH, "seq", MAX_SEQ_LEN, "budget", f"{TOKEN_BUDGET:,}", "resume", RESUME)
print("wall_clock_stop_h", WALL_CLOCK_HOURS)


### Unsloth — download + load (`FastLanguageModel`, BF16)


In [ ]:
# Text-only Causal LM path — required for Unsloth sample packing.
import os
os.environ.setdefault("UNSLOTH_STABLE_DOWNLOADS", "1")
os.environ.setdefault("UNSLOTH_DISABLE_STATISTICS", "1")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.pop(k, None)

from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel

def local_model_complete(path: Path) -> bool:
    if not path.is_dir() or not (path / "config.json").is_file():
        return False
    for pattern in ("*.safetensors", "pytorch_model*.bin", "model*.bin"):
        for p in path.glob(pattern):
            if p.is_file() and p.stat().st_size > 1_000_000:
                return True
    return False

if local_model_complete(MODEL_DIR):
    print("Using cached model:", MODEL_DIR)
else:
    if MODEL_DIR.exists():
        print("Incomplete local dir; removing", MODEL_DIR)
        shutil.rmtree(MODEL_DIR, ignore_errors=True)
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    print("Prefetching", MODEL_NAME, "→", MODEL_DIR)
    snapshot_download(
        repo_id=MODEL_NAME,
        local_dir=str(MODEL_DIR),
        token=HF_TOKEN,
        resume_download=True,
        max_workers=8,
    )
    assert local_model_complete(MODEL_DIR), f"Prefetch incomplete: {MODEL_DIR}"
    gb = sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file()) / 1e9
    print(f"Prefetch complete ({gb:.1f} GB)")

load_kwargs = dict(
    model_name=str(MODEL_DIR),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto bf16 on A100
    load_in_4bit=False,  # Phase 1 = BF16 LoRA
)
try:
    model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
except TypeError:
    load_kwargs.pop("max_seq_length", None)
    model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)

cfg = model.config
archs = list(getattr(cfg, "architectures", None) or [])
has_vision = hasattr(cfg, "vision_config")
print("Loaded", MODEL_NAME, "| tokenizer", type(tokenizer).__name__, "| load_in_4bit=False")
print("architectures:", archs, "| vision_config:", has_vision)
assert not has_vision, "Model has vision_config — Unsloth will skip packing. Use a text-only base."
assert not any(a.endswith("ForConditionalGeneration") for a in archs), (
    f"ConditionalGeneration architecture {archs} → VLM packing skip. Wrong checkpoint."
)


### LoRA (attention + MLP)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
print("LoRA ready: r=", LORA_R, "alpha=", LORA_ALPHA)


<a name="Data"></a>
### Data Prep

Streaming CPT corpus (`text` field). **No** chat template.


In [ ]:
from datasets import load_dataset

def keep_text(example):
    text = example.get("text") or ""
    return {"text": text if isinstance(text, str) else ""}

train_dataset = load_dataset(DATASET, split="train", streaming=True)
train_dataset = train_dataset.shuffle(seed=SEED, buffer_size=10_000)
try:
    cols = list(train_dataset.column_names or [])
except Exception:
    cols = []
drop = [c for c in cols if c != "text"]
train_dataset = train_dataset.map(keep_text, remove_columns=drop) if drop else train_dataset.map(keep_text)
train_dataset = train_dataset.filter(lambda ex: bool(ex["text"] and ex["text"].strip()))

eval_dataset = None
for split in ("validation", "val"):
    try:
        eval_dataset = load_dataset(DATASET, split=split, streaming=True)
        eval_dataset = eval_dataset.map(keep_text).filter(lambda ex: bool(ex["text"] and ex["text"].strip()))
        print("val split:", split)
        break
    except Exception:
        pass

row = next(iter(train_dataset.take(1)))
print("sample chars:", len(row["text"]), "| head:", repr(row["text"][:120]))


<a name="Train"></a>
### Train

`packing=True` is required. Notebook **aborts** if Unsloth skips packing.


In [ ]:
from transformers import TrainerCallback, TrainerControl, TrainerState, TrainingArguments
from trl import SFTConfig, SFTTrainer

@dataclass
class Phase1State:
    tokens_seen: int = 0
    global_step: int = 0
    tok_per_s: float = 0.0
    best_loss: float = float("inf")

def write_latest(root: Path, ckpt: Path):
    root.mkdir(parents=True, exist_ok=True)
    (root / "LATEST").write_text(str(ckpt.resolve()))

def read_latest(root: Path):
    f = root / "LATEST"
    if not f.exists():
        return None
    p = Path(f.read_text().strip())
    return p if p.exists() else None

def mirror_to_drive(src: Path):
    if DRIVE_CKPT is None:
        return
    dest = DRIVE_CKPT / src.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest, dirs_exist_ok=True)
    write_latest(DRIVE_CKPT, dest)
    print("Mirrored →", dest, flush=True)

def resolve_resume():
    if RESUME in ("none", "", "False", "false"):
        return None
    if RESUME != "auto":
        p = Path(RESUME)
        assert p.exists(), p
        return str(p)
    for root in filter(None, [DRIVE_CKPT, OUT_DIR]):
        latest = read_latest(root)
        if latest is not None:
            print("Resume auto →", latest)
            return str(latest)
        ckpts = sorted(Path(root).glob("checkpoint-*"), key=lambda p: p.stat().st_mtime)
        if ckpts:
            print("Resume auto →", ckpts[-1])
            return str(ckpts[-1])
    print("Resume auto: fresh start")
    return None

def estimate_max_steps():
    world = max(int(os.environ.get("WORLD_SIZE", "1")), 1)
    tps = BATCH * ACCUM * MAX_SEQ_LEN * world
    return max(int(math.ceil(TOKEN_BUDGET / max(tps, 1))) + 100, 100)

class TokenBudgetCallback(TrainerCallback):
    def __init__(self, state: Phase1State):
        self.ps = state
        self.t0 = time.time()
        self.last_t = self.t0
        self.last_tok = state.tokens_seen
        self.last_save = self.t0
        self.early_done = False
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def _tps(self, args):
        world = max(int(os.environ.get("WORLD_SIZE", "1")), 1)
        return args.per_device_train_batch_size * args.gradient_accumulation_steps * MAX_SEQ_LEN * world

    def _vram(self):
        if not torch.cuda.is_available():
            return 0.0, 0.0
        return torch.cuda.memory_allocated() / 1e9, torch.cuda.max_memory_allocated() / 1e9

    def on_step_end(self, args, state, control, **kwargs):
        self.ps.tokens_seen += self._tps(args)
        self.ps.global_step = state.global_step
        now = time.time()
        dt = now - self.last_t
        if dt >= 30 or state.global_step % max(args.logging_steps, 1) == 0:
            tok_s = (self.ps.tokens_seen - self.last_tok) / max(dt, 1e-6)
            self.ps.tok_per_s = tok_s
            remain = max(TOKEN_BUDGET - self.ps.tokens_seen, 0)
            eta_h = (remain / max(tok_s, 1.0)) / 3600.0
            alloc, peak = self._vram()
            loss = "n/a"
            if state.log_history:
                v = state.log_history[-1].get("loss")
                if isinstance(v, (int, float)):
                    loss = f"{v:.4f}"
            print(
                f"step={state.global_step} tokens={self.ps.tokens_seen:,} tok/s={tok_s:.0f} "
                f"eta_budget={eta_h:.1f}h loss={loss} VRAM={alloc:.1f}G peak={peak:.1f}G",
                flush=True,
            )
            self.last_t, self.last_tok = now, self.ps.tokens_seen
        if not self.early_done and (now - self.t0) >= PROJECT_AFTER_MINUTES * 60:
            self.early_done = True
            tok_s = self.ps.tok_per_s or (self.ps.tokens_seen / max(now - self.t0, 1))
            proj = tok_s * 3600 * REMAINING_COLAB_HOURS
            _, peak = self._vram()
            print(
                f"EARLY_PROJECTION: tok/s≈{tok_s:.0f} → ~{proj/1e9:.2f}B over {REMAINING_COLAB_HOURS:.0f}h "
                f"(target {TOKEN_BUDGET/1e9:.2f}B) | peak={peak:.1f}G | "
                f"raise BATCH if peak<<VRAM; cut BATCH on OOM; wall_stop={WALL_CLOCK_HOURS}h",
                flush=True,
            )
        if (now - self.last_save) >= CKPT_MINUTES * 60:
            control.should_save = True
            self.last_save = now
            print(f"Timed checkpoint ({CKPT_MINUTES:.0f} min)", flush=True)
        if self.ps.tokens_seen >= TOKEN_BUDGET:
            print("Token budget reached; stopping.", flush=True)
            control.should_training_stop = True
            control.should_save = True
        if WALL_CLOCK_HOURS and WALL_CLOCK_HOURS > 0:
            elapsed_h = (now - self.t0) / 3600.0
            if elapsed_h >= WALL_CLOCK_HOURS:
                print(
                    f"Wall-clock limit {WALL_CLOCK_HOURS:.1f}h reached "
                    f"(elapsed {elapsed_h:.2f}h); saving and stopping before Colab kill.",
                    flush=True,
                )
                control.should_training_stop = True
                control.should_save = True
        return control

    def on_save(self, args, state, control, **kwargs):
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if ckpt.exists():
            (ckpt / "phase1_state.json").write_text(json.dumps(asdict(self.ps), indent=2))
            write_latest(OUT_DIR, ckpt)
            mirror_to_drive(ckpt)
        if state.log_history:
            v = state.log_history[-1].get("loss")
            if isinstance(v, (int, float)) and v < self.ps.best_loss:
                self.ps.best_loss = float(v)
        return control

phase_state = Phase1State()
resume_path = resolve_resume()
if resume_path:
    sf = Path(resume_path) / "phase1_state.json"
    if sf.exists():
        d = json.loads(sf.read_text())
        phase_state = Phase1State(**{k: d[k] for k in Phase1State.__dataclass_fields__ if k in d})
        print("Restored tokens_seen=", f"{phase_state.tokens_seen:,}", "step=", phase_state.global_step)
    if phase_state.tokens_seen >= TOKEN_BUDGET:
        raise SystemExit("Token budget already met; nothing to do.")

max_steps = MAX_STEPS if MAX_STEPS is not None else estimate_max_steps()
print("max_steps=", max_steps, "token_budget=", f"{TOKEN_BUDGET:,}")

optim = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch"
sft_kwargs = dict(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=5 if SMOKE else 100,
    weight_decay=0.01,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    max_steps=max_steps,
    bf16=True,
    optim=optim,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    packing=PACKING,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
)
if HUB_MODEL_ID and HF_TOKEN:
    sft_kwargs.update(
        push_to_hub=True,
        hub_model_id=HUB_MODEL_ID,
        hub_strategy="every_save",
        hub_private_repo=True,
    )

def build_sft_config(kwargs):
    try:
        return SFTConfig(**kwargs)
    except TypeError:
        kwargs = dict(kwargs)
        if "max_seq_length" in kwargs:
            kwargs["max_length"] = kwargs.pop("max_seq_length")
            try:
                return SFTConfig(**kwargs)
            except TypeError:
                pass
        for k in ("packing", "dataset_text_field", "logging_first_step", "max_length", "max_seq_length"):
            kwargs.pop(k, None)
        return SFTConfig(**kwargs)

args = build_sft_config(sft_kwargs)

trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    callbacks=[TokenBudgetCallback(phase_state)],
)
if eval_dataset is not None:
    trainer_kwargs["eval_dataset"] = eval_dataset

try:
    trainer = SFTTrainer(**trainer_kwargs)
except TypeError:
    trainer_kwargs.pop("processing_class", None)
    trainer_kwargs["tokenizer"] = tokenizer
    trainer = SFTTrainer(**trainer_kwargs)

packing_on = bool(getattr(trainer.args, "packing", False))
if PACKING and not packing_on:
    raise SystemExit(
        "FATAL: packing requested but trainer.args.packing is False. "
        "Do not burn credits. Confirm FastLanguageModel + text-only Causal LM "
        "(no vision_config / ForConditionalGeneration)."
    )
print("Sample packing is ACTIVE" if packing_on else "packing OFF (allowed only if PACKING=False)")


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint=resume_path)


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")
print("tokens_seen=", f"{phase_state.tokens_seen:,}", "tok/s≈", f"{phase_state.tok_per_s:.0f}")


<a name="Inference"></a>
### Quick text smoke (optional)


In [ ]:
FastLanguageModel.for_inference(model)
prompt = "def fibonacci(n):\n    "
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(**inputs, streamer=streamer, max_new_tokens=128, use_cache=True)


<a name="Save"></a>
### Save LoRA adapters (local + Drive + Hub)


In [ ]:
final_dir = OUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
(final_dir / "phase1_state.json").write_text(json.dumps(asdict(phase_state), indent=2))
write_latest(OUT_DIR, final_dir)
mirror_to_drive(final_dir)
print("Saved", final_dir)

if HUB_MODEL_ID and HF_TOKEN:
    model.push_to_hub(HUB_MODEL_ID, private=True, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_MODEL_ID, private=True, token=HF_TOKEN)
    print("Pushed adapters →", HUB_MODEL_ID)


### Next session

1. Re-run **Install** + **Config** with `SMOKE = False`, `RESUME = "auto"`.
2. Confirm smoke logged **`Sample packing is ACTIVE`** before full CPT.
3. Full run defaults: `BATCH=110` on A100 80GB; cut on OOM. Wall-clock stop at **11.5h** auto-saves then stops.
4. Confirm Drive mirror under `coder-qwen25-coder-7b-phase1-lora/` before session end.

See `fine-tune/FINE_TUNE_DECISIONS.md`.
